# Day 2 — Feature Engineering & Target Construction
YouTube Real-Time Engagement Prediction Project

Goal for today:
1. Load Day 1's cleaned data (one row per snapshot)
2. For each video, extract EARLY-WINDOW features (from snapshots <= 6h old)
3. For each video, construct the TARGET (engagement_rate closest to 24h,
   accepted within an 18-30h window)
4. Merge into ONE row per video — this becomes the actual ML training table
5. Save it for Day 3 (modeling)

NOTE: Title-based features are intentionally EXCLUDED. With only ~160
unique videos, text-derived features carry a high risk of overfitting to
sample-specific patterns rather than learning generalizable relationships.
Numeric/categorical features are preferred for their statistical reliability
at this sample size.

In [1]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)

PROCESSED_PATH = "C:/Users/Lenovo/youtube_engagement_project/data/processed/cleaned_data.csv"
TRAINING_TABLE_PATH = "C:/Users/Lenovo/youtube_engagement_project/data/processed/training_table.csv"

EARLY_CUTOFF = 6          # hours — features built from snapshots at or before this age
TARGET_LOW, TARGET_HIGH = 18, 30   # hours — window accepted as "~24h" target

df = pd.read_csv(PROCESSED_PATH)
df["published_at"] = pd.to_datetime(df["published_at"], utc=True)
df["snapshot_time"] = pd.to_datetime(df["snapshot_time"], utc=True)

print("Shape:", df.shape)
print("Unique videos:", df["video_id"].nunique())

Shape: (41487, 16)
Unique videos: 162


## 1. Time-based features (from published_at — known at upload time)
These are static per video, so we can compute them directly on the full df
then just take one value per video later.

In [2]:
df["upload_hour"] = df["published_at"].dt.hour
df["upload_dayofweek"] = df["published_at"].dt.dayofweek       # 0=Monday
df["is_weekend"] = df["upload_dayofweek"].isin([5, 6]).astype(int)

## 2. Build EARLY-WINDOW features (one row per video)
For each video, take the snapshot CLOSEST TO (but not exceeding) the early
cutoff — this gives us the most information available while still being
"early". This snapshot's views/likes/comments/rates become our features.

In [3]:
early_df = df[df["video_age_hours"] <= EARLY_CUTOFF].copy()

# For each video, pick the row with the LARGEST video_age_hours within the
# early window (i.e. the most recent snapshot still inside the early period)
early_idx = early_df.groupby("video_id")["video_age_hours"].idxmax()
early_features = early_df.loc[early_idx].copy()

# Rename to make clear these are "early" values
early_features = early_features.rename(columns={
    "views": "early_views",
    "likes": "early_likes",
    "comments": "early_comments",
    "views_per_hour": "early_views_per_hour",
    "like_rate": "early_like_rate",
    "comment_rate": "early_comment_rate",
    "engagement_rate": "early_engagement_rate",
    "video_age_hours": "early_snapshot_age_hours",
})

# Count how many early snapshots each video had (a simple data-density feature)
early_snapshot_counts = early_df.groupby("video_id").size().rename("early_snapshot_count")

early_features = early_features.merge(early_snapshot_counts, on="video_id")

# Keep only the columns we need from the early snapshot
early_cols = [
    "video_id", "category_id", "duration_seconds",
    "upload_hour", "upload_dayofweek", "is_weekend",
    "early_snapshot_age_hours", "early_snapshot_count",
    "early_views", "early_likes", "early_comments",
    "early_views_per_hour", "early_like_rate", "early_comment_rate",
    "early_engagement_rate",
]
early_features = early_features[early_cols]

print("Early features shape:", early_features.shape)
early_features.head()

Early features shape: (162, 15)


,video_id,category_id,duration_seconds,upload_hour,upload_dayofweek,is_weekend,early_snapshot_age_hours,early_snapshot_count,early_views,early_likes,early_comments,early_views_per_hour,early_like_rate,early_comment_rate,early_engagement_rate
0,0Bxd2gEZTh0,24.0,299,9,2,0,5.908491,7,36802.0,1203.0,32.0,6228.662870,3.268844,0.086952,3.355796
1,0G1bThtvAXs,10.0,225,0,6,1,5.602963,1,55702.0,9049.0,975.0,9941.525701,16.245377,1.750386,17.995763
2,0Q0Yb4Livuw,24.0,564,12,1,0,5.950920,15,29468.0,1544.0,52.0,4951.839588,5.239582,0.176463,5.416045
3,0lqacLeyMcI,20.0,4622,4,6,1,1.604352,1,1512147.0,4286.0,0.0,942528.303100,0.283438,0.000000,0.283438
4,1Vqp6szqqMY,20.0,21995,17,6,1,5.946545,21,711937.0,11428.0,178.0,119722.805400,1.605198,0.025002,1.630200


## 3. Build the TARGET — engagement_rate closest to 24h
For each video, find the snapshot within the 18-30h window whose
video_age_hours is closest to 24. Use its engagement_rate as the target.

In [4]:
target_window_df = df[(df["video_age_hours"] >= TARGET_LOW) &
                       (df["video_age_hours"] <= TARGET_HIGH)].copy()

target_window_df["distance_from_24h"] = (target_window_df["video_age_hours"] - 24).abs()

target_idx = target_window_df.groupby("video_id")["distance_from_24h"].idxmin()
target_rows = target_window_df.loc[target_idx].copy()

target_rows = target_rows.rename(columns={
    "engagement_rate": "target_engagement_rate",
    "video_age_hours": "target_snapshot_age_hours",
})

target_cols = ["video_id", "target_snapshot_age_hours", "target_engagement_rate"]
target_rows = target_rows[target_cols]

print("Target rows shape:", target_rows.shape)
target_rows.head()

Target rows shape: (162, 3)


,video_id,target_snapshot_age_hours,target_engagement_rate
114,0Bxd2gEZTh0,23.940590,1.602503
594,0G1bThtvAXs,24.028963,8.254243
959,0Q0Yb4Livuw,23.979707,4.201912
1070,0lqacLeyMcI,24.034667,0.264976
1231,1Vqp6szqqMY,21.373108,1.607400


## 4. Merge features + target into ONE training table (one row per video)

In [5]:
training_table = early_features.merge(target_rows, on="video_id", how="inner")

print("Final training table shape:", training_table.shape)
print("Unique videos:", training_table["video_id"].nunique())
training_table.head()

Final training table shape: (162, 17)
Unique videos: 162


,video_id,category_id,duration_seconds,upload_hour,upload_dayofweek,is_weekend,early_snapshot_age_hours,early_snapshot_count,early_views,early_likes,early_comments,early_views_per_hour,early_like_rate,early_comment_rate,early_engagement_rate,target_snapshot_age_hours,target_engagement_rate
0,0Bxd2gEZTh0,24.0,299,9,2,0,5.908491,7,36802.0,1203.0,32.0,6228.662870,3.268844,0.086952,3.355796,23.940590,1.602503
1,0G1bThtvAXs,10.0,225,0,6,1,5.602963,1,55702.0,9049.0,975.0,9941.525701,16.245377,1.750386,17.995763,24.028963,8.254243
2,0Q0Yb4Livuw,24.0,564,12,1,0,5.950920,15,29468.0,1544.0,52.0,4951.839588,5.239582,0.176463,5.416045,23.979707,4.201912
3,0lqacLeyMcI,20.0,4622,4,6,1,1.604352,1,1512147.0,4286.0,0.0,942528.303100,0.283438,0.000000,0.283438,24.034667,0.264976
4,1Vqp6szqqMY,20.0,21995,17,6,1,5.946545,21,711937.0,11428.0,178.0,119722.805400,1.605198,0.025002,1.630200,21.373108,1.607400


## 5. Sanity check — confirm no leakage columns remain
We should NOT have: views, likes, comments, like_rate, comment_rate,
engagement_rate (the raw/current-time versions) as features — only the
"early_" prefixed versions and the target are allowed.

In [6]:
leakage_risk_cols = ["views", "likes", "comments", "like_rate", "comment_rate", "engagement_rate"]
present = [c for c in leakage_risk_cols if c in training_table.columns]
print("Leakage-risk columns present (should be empty):", present)

Leakage-risk columns present (should be empty): []


## 6. Quick check of target distribution

In [7]:
print(training_table["target_engagement_rate"].describe())

count    162.000000
mean       3.686902
std        3.125555
min        0.000000
25%        1.395584
50%        3.086841
75%        5.322004
max       20.037423
Name: target_engagement_rate, dtype: float64


## 7. Save the final training table for Day 3 (modeling)

In [8]:
import os
os.makedirs(os.path.dirname(TRAINING_TABLE_PATH), exist_ok=True)
training_table.to_csv(TRAINING_TABLE_PATH, index=False)
print(f"Saved training table to {TRAINING_TABLE_PATH}")
print(training_table.shape)

Saved training table to C:/Users/Lenovo/youtube_engagement_project/data/processed/training_table.csv
(162, 17)


In [9]:
training_table.shape

(162, 17)

In [12]:
training_table.head()

,video_id,category_id,duration_seconds,upload_hour,upload_dayofweek,is_weekend,early_snapshot_age_hours,early_snapshot_count,early_views,early_likes,early_comments,early_views_per_hour,early_like_rate,early_comment_rate,early_engagement_rate,target_snapshot_age_hours,target_engagement_rate
0,0Bxd2gEZTh0,24.0,299,9,2,0,5.908491,7,36802.0,1203.0,32.0,6228.662870,3.268844,0.086952,3.355796,23.940590,1.602503
1,0G1bThtvAXs,10.0,225,0,6,1,5.602963,1,55702.0,9049.0,975.0,9941.525701,16.245377,1.750386,17.995763,24.028963,8.254243
2,0Q0Yb4Livuw,24.0,564,12,1,0,5.950920,15,29468.0,1544.0,52.0,4951.839588,5.239582,0.176463,5.416045,23.979707,4.201912
3,0lqacLeyMcI,20.0,4622,4,6,1,1.604352,1,1512147.0,4286.0,0.0,942528.303100,0.283438,0.000000,0.283438,24.034667,0.264976
4,1Vqp6szqqMY,20.0,21995,17,6,1,5.946545,21,711937.0,11428.0,178.0,119722.805400,1.605198,0.025002,1.630200,21.373108,1.607400


In [13]:
training_table.dtypes

video_id                         str
category_id                  float64
duration_seconds               int64
upload_hour                    int32
upload_dayofweek               int32
is_weekend                     int64
early_snapshot_age_hours     float64
early_snapshot_count           int64
early_views                  float64
early_likes                  float64
early_comments               float64
early_views_per_hour         float64
early_like_rate              float64
early_comment_rate           float64
early_engagement_rate        float64
target_snapshot_age_hours    float64
target_engagement_rate       float64
dtype: object

In [14]:
training_table.isnull().sum()

video_id                     0
category_id                  0
duration_seconds             0
upload_hour                  0
upload_dayofweek             0
is_weekend                   0
early_snapshot_age_hours     0
early_snapshot_count         0
early_views                  0
early_likes                  0
early_comments               0
early_views_per_hour         0
early_like_rate              0
early_comment_rate           0
early_engagement_rate        0
target_snapshot_age_hours    0
target_engagement_rate       0
dtype: int64

In [15]:
training_table['target_engagement_rate'].describe()

count    162.000000
mean       3.686902
std        3.125555
min        0.000000
25%        1.395584
50%        3.086841
75%        5.322004
max       20.037423
Name: target_engagement_rate, dtype: float64